In [1]:
import pandas as pd
df = pd.read_parquet("/kaggle/input/datasets/manarabuirsheid/news-bias-data/snn.parquet")
print(df.shape)

(30050, 11)


In [2]:
df["source"].value_counts()

source
alriyadh        4754
aljazirah       3274
alyaum          3038
aawsat          2943
okaz            2821
aleqtisadiya    2808
alwatan         2267
almadina        2138
alweeam         2072
3alyoum         2071
sabq            1368
was              366
arreyadi          80
arriyadiyah       50
Name: count, dtype: int64

In [3]:
a = df[df["source"] == "alriyadh"].reset_index(drop=True)
b = df[df["source"] == "aljazirah"].reset_index(drop=True)
print(len(a), len(b))

4754 3274


In [4]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
print("الموديل جاهز")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

الموديل جاهز


In [14]:
a_texts = (a["title_clean"].fillna("") + " " + a["text_clean"].fillna("").str[:500]).tolist()
b_texts = (b["title_clean"].fillna("") + " " + b["text_clean"].fillna("").str[:500]).tolist()

emb_a = model.encode(a_texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
emb_b = model.encode(b_texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)

print(emb_a.shape, emb_b.shape)

Batches:   0%|          | 0/75 [00:00<?, ?it/s]

Batches:   0%|          | 0/52 [00:00<?, ?it/s]

(4754, 384) (3274, 384)


In [16]:
import numpy as np

sim = emb_a @ emb_b.T
best_j = sim.argmax(axis=1)
best_score = sim.max(axis=1)

pairs = pd.DataFrame({"a_idx": range(len(a)), "b_idx": best_j, "score": best_score})
pairs = pairs.sort_values("score", ascending=False)

for _, r in pairs.head(5).iterrows():
    print(round(r["score"], 2))
    print("الرياض:", a.loc[r["a_idx"], "title"])
    print("الجزيرة:", b.loc[r["b_idx"], "title"])
    print("-----")

1.0
الرياض: رصد ذئب رمادي في كاليفورنيا للمرة الثانية منذ عام 1924
الجزيرة: رصد ذئب رمادي في كاليفورنيا للمرة الثانية منذ عام 1924
-----
1.0
الرياض: «الغذاء والدواء» تجري مسحاً ل«الهيدروكربونات العطرية» في الأغذية
الجزيرة: «الغذاء والدواء» تجري مسحاً لـ«الهيدروكربونات العطرية» في الأغذية
-----
1.0
الرياض: القتل تعزيراً لمهرب هيروين بثادق
الجزيرة: القتل تعزيراً لمهرب هيروين بثادق
-----
1.0
الرياض: إغلاق طريق الملك عبدالعزيز والشارع العاشر بالخبر.. اليوم
الجزيرة: إغلاق طريق الملك عبد العزيز والشارع العاشر بالخبر.. اليوم
-----
1.0
الرياض: والد الزميل حمود العمار إلى رحمة الله
الجزيرة: والد الزميل حمود العمار إلى رحمه الله
-----


In [17]:
print(pd.cut(pairs["score"], bins=[0, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 1.01]).value_counts().sort_index())

for low, high in [(0.9, 0.95), (0.8, 0.9), (0.7, 0.8), (0.6, 0.7)]:
    print("\n========== بين", low, "و", high, "==========")
    sample = pairs[(pairs["score"] >= low) & (pairs["score"] < high)].sample(3, random_state=1)
    for _, r in sample.iterrows():
        print(round(r["score"], 2))
        print("الرياض:", a.loc[r["a_idx"], "title"])
        print("الجزيرة:", b.loc[r["b_idx"], "title"])
        print("-----")
        

score
(0.0, 0.5]       128
(0.5, 0.6]       356
(0.6, 0.7]       607
(0.7, 0.8]       869
(0.8, 0.9]       948
(0.9, 0.95]      521
(0.95, 1.01]    1325
Name: count, dtype: int64

========== بين 0.9 و 0.95 ==========
0.93
الرياض: «الدار دارك»   في ضباء
الجزيرة: أمير منطقة عسير يرعى ليالي المحافظات بالساحة الشعبية
-----
0.93
الرياض: الإرهابيون الجدد في «تويتر»!
الجزيرة: أمير منطقة نجران: الإرهابيون شرذمة بغيضة انتهكت حرمة بيت الله
-----
0.92
الرياض: الشعبي: الرائد أغلق ملف الأجانب بالغيني زياتي
الجزيرة: المرشدي يلتحق ببعثة الشباب في هولندا
-----

========== بين 0.8 و 0.9 ==========
0.88
الرياض: حريق في كوبري الخرج الجديد بسبب التماس كهربائي
الجزيرة: حادث حريق بحي الفهد الشمالي بمنطقة نجران
-----
0.81
الرياض: وزير التعليم يزور جبال فيفاء ويناقش خطة بدء العام الدراسي
الجزيرة: وكيل إمارة الباحة يناقش القيادات التعليمية العام الدراسي الجديد
-----
0.86
الرياض: الحكومة اليمنية تعمل على إعادة فتح مطار عدن قريباً
الجزيرة: فتح مطار عدن بشرى خير لإعادة استقرار اليمن
-----

========== بين 0.7 و 0.

In [18]:
da = pd.to_datetime(a["date"]).values.astype("datetime64[D]")
db = pd.to_datetime(b["date"]).values.astype("datetime64[D]")
diff = np.abs((da[:, None] - db[None, :]).astype(int))

sim2 = np.where(diff <= 2, sim, -1)

best_j2 = sim2.argmax(axis=1)
best_score2 = sim2.max(axis=1)
pairs2 = pd.DataFrame({"a_idx": range(len(a)), "b_idx": best_j2, "score": best_score2})

print(pd.cut(pairs2["score"], bins=[-1.1, 0, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 1.01]).value_counts().sort_index())

for low, high in [(0.85, 0.95), (0.75, 0.85)]:
    print("\n========== بين", low, "و", high, "==========")
    sample = pairs2[(pairs2["score"] >= low) & (pairs2["score"] < high)].sample(4, random_state=2)
    for _, r in sample.iterrows():
        print(round(r["score"], 2))
        print("الرياض:", a.loc[r["a_idx"], "title"])
        print("الجزيرة:", b.loc[r["b_idx"], "title"])
        print("-----")

score
(-1.1, 0.0]        0
(0.0, 0.5]       365
(0.5, 0.6]       477
(0.6, 0.7]       681
(0.7, 0.8]       728
(0.8, 0.9]       753
(0.9, 0.95]      497
(0.95, 1.01]    1253
Name: count, dtype: int64

========== بين 0.85 و 0.95 ==========
0.92
الرياض: الأمطار  تؤجل ليالي محافظات أبها
الجزيرة: معرض تشكيلي لإنجازات المليك
-----
0.89
الرياض: سعد بن هذلول يحتفل بزواجه من كريمة عبدالرحمن آل حامد
الجزيرة: شقراء تحتفل بعيد الفطر المبارك
-----
0.95
الرياض: بدء تسيير الرحلات المدنية من مطار عدن الدولي
الجزيرة: مطار عدن الدولي يستأنف رحلاته بعد توقف أربعة أشهر
-----
0.9
الرياض: وزير داخلية البحرين: إيران متورطة في الإخلال بأمننا وصدّرت إلينا ثقافة الإرهاب
الجزيرة: البحرين ترفض قطعياً التدخلات الإيرانية في شؤونها الداخلية
-----

========== بين 0.75 و 0.85 ==========
0.79
الرياض: «التعليم» يعتمد زي الطالبات المدرسي
الجزيرة: تعليم مكة المكرمة يطبق الزي المدرسي للبنات في المراحل الثلاث
-----
0.78
الرياض: القوات العراقية تستعيد جامعة الأنبار من (داعش)
الجزيرة: تقدم ملحوظ للقوات العراقية في الأنبار
--

In [19]:
from transformers import pipeline

ner = pipeline("ner", model="CAMeL-Lab/bert-base-arabic-camelbert-mix-ner",
               aggregation_strategy="simple", device=0)

print(ner(a.loc[0, "title"]))
print(ner("الحكومة اليمنية تعمل على إعادة فتح مطار عدن قريباً"))

config.json:   0%|          | 0.00/980 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  436MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  436MB            

...

model.safetensors: downloading bytes:           |  0.00B            

tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/305k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[{'entity_group': 'LOC', 'score': np.float32(0.9923948), 'word': 'الرياض', 'start': 11, 'end': 17}, {'entity_group': 'PERS', 'score': np.float32(0.99139625), 'word': 'مرزوق الهيضل', 'start': 23, 'end': 35}]
[{'entity_group': 'LOC', 'score': np.float32(0.95860535), 'word': 'عدن', 'start': 40, 'end': 43}]


In [20]:
def get_ents(df_part):
    texts = (df_part["title"].fillna("") + " " + df_part["text_light"].fillna("").str[:300]).tolist()
    results = ner(texts, batch_size=32)
    return [set(e["word"] for e in r if len(e["word"]) > 1) for r in results]

a["ents"] = get_ents(a)
b["ents"] = get_ents(b)

for i in range(3):
    print(a.loc[i, "title"])
    print(a.loc[i, "ents"])
    print("-----")

أمير منطقة الرياض يقلد مرزوق الهيضل رتبة عقيد
{'مرزوق بن سلطان الهيضل', 'مرزوق الهيضل', 'فيصل بن بندر بن عبدالعزيز', 'الرياض', 'الهيضل'}
-----
«التعليم» و«تعليم الرياض» تعايدان منسوبيها.. غداً
{'عزام بن محمد الدخيل', 'الرياض', 'تعليم', 'منصور الحسين'}
-----
فهد الفرج إلى رحمة الله
{'فهد الفرج', '##جد', 'فهد بن فرج بن عبدالله الفرج', 'الملك عبدالعزيز', 'الخرج', 'خالد', 'الخزام', 'الموسى', '##ليم', 'الله', 'وما', 'وعبدالله وعبدالرحمن'}
-----


In [21]:
from collections import Counter

def clean(s):
    return {w for w in s if not w.startswith("##") and len(w) > 2}

a["ents"] = a["ents"].apply(clean)
b["ents"] = b["ents"].apply(clean)

cnt = Counter(w for s in list(a["ents"]) + list(b["ents"]) for w in s)
common = {w for w, c in cnt.items() if c > 150}

print(len(common))
print(sorted(common, key=lambda w: -cnt[w]))

8
['الله', 'الرياض', 'عسير', 'ريال', 'مصر', 'داعش', 'بالرياض', 'مكة']


In [23]:
def shared_names(r):
    ea = a.loc[r["a_idx"], "ents"] - common
    eb = b.loc[r["b_idx"], "ents"] - common
    return ea & eb

pairs2["shared"] = pairs2.apply(shared_names, axis=1)
pairs2["n_shared"] = pairs2["shared"].apply(len)

def label(r):
    if r["score"] >= 0.95:
        return "نسخة"
    if r["score"] >= 0.75 and r["n_shared"] >= 1:
        return "نفس الحدث"
    return "مرفوض"

pairs2["type"] = pairs2.apply(label, axis=1)
print(pairs2["type"].value_counts())

sample = pairs2[pairs2["type"] == "نفس الحدث"].sample(6, random_state=3)
for _, r in sample.iterrows():
    print(round(r["score"], 2), r["shared"])
    print("الرياض:", a.loc[r["a_idx"], "title"])
    print("الجزيرة:", b.loc[r["b_idx"], "title"])
    print("-----")

type
مرفوض        2883
نسخة         1253
نفس الحدث     618
Name: count, dtype: int64
0.89 {'موبايلي'}
الرياض: «موبايلي» تخسر 945 مليون ريال في النصف الأول.. وهيئة السوق تعيدها للتداول.. اليوم
الجزيرة: موبايلي: اتبعنا سياسة محاسبية متحفظة
-----
0.8 {'عدن'}
الرياض: طائرة سعودية ثانية محملة ب 10 أطنان من المساعدات تصل عدن
الجزيرة: أول طائرة إغاثية سعودية تهبط في مطار عدن
-----
0.77 {'أحد', 'والأن'}
الرياض: أحد والأنصار يحتجان على موعد «دربي المدينة»
الجزيرة: الاتفاق على إقامة لقاء كروي بين قطبي المدينة أحد والأنصار
-----
0.81 {'إيران'}
الرياض: إيران ستستفيد من الاتفاق النووي للاستمرار في عبثها بشؤون المنطقة
الجزيرة: لا حل مع إيران إلا مواجهة التصعيد بالتصعيد
-----
0.93 {'المملكة العربية السعودية', 'لوس أنجلوس'}
الرياض: نجوم المملكة يحرزون 15 ميدالية في الأولمبياد الخاص بلوس أنجليس
الجزيرة: نجوم الأخضر يرفعون الغلة السعودية إلى 15 ميدالية
-----
0.94 {'بغداد'}
الرياض: وزير الدفاع الأميركي يبحث في بغداد مواجهة «داعش»
الجزيرة: وزير الدفاع الأميركي يصل إلى بغداد لبحث مواجهة تنظيم داعش
-----
